# Day 16 — Custom Ag Object Detection

**Agri-AI Bootcamp** · Phase 4 of 5: *Object Detection & Yield* · Days 13–16

> Download a Roboflow fruit/veggie dataset (YOLO format), train YOLOv8n for 20 epochs, estimate counts.

**Runtime:** `Runtime -> Change runtime type -> T4 GPU` before you start.

**How to use this notebook:** run the cells top to bottom. The *Worked solution*
section is complete and runnable; the *Your turn* section at the end is where you
experiment. Datasets that need a Kaggle/Roboflow key are simulated here with the
same columns and shapes, and the real download command is shown in a comment —
swap it in when you have the credentials. Example numbers are illustrative.

## 1. Explained

Custom detection needs labelled images and a data.yaml file listing train/val folders and class names. Fine-tuning starts from pre-trained weights, so 20 epochs on a small dataset can already work. Counting detections per image gives a simple yield proxy.

### 🌾 Agriculture example

On a sweet-pepper dataset, count boxes per plant image to compare plots, then calibrate counts against harvested fruit.

> **Watch out:** Counts miss hidden fruit; treat them as estimates and validate on hand-counted images.

## 2. Setup

In [ ]:
!pip -q install ultralytics

In [ ]:
from ultralytics import YOLO
import numpy as np, os, glob, yaml, shutil
from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import torch
print("CUDA:", torch.cuda.is_available(), "| set Runtime -> T4 GPU before training")

## 3. Data

In [ ]:
# ---- Build a tiny YOLO-format dataset ---------------------------------------
# REAL WORKFLOW (use this once you have a Roboflow API key):
#   !pip -q install roboflow
#   from roboflow import Roboflow
#   rf = Roboflow(api_key="YOUR_KEY")
#   ds = rf.workspace("...").project("...").version(1).download("yolov8")
#   DATA_YAML = ds.location + "/data.yaml"
#
# For this notebook we SYNTHESISE a pepper-counting dataset so every cell runs.
root = Path("pepper_ds")
if root.exists(): shutil.rmtree(root)
for split in ["train", "val"]:
    (root/split/"images").mkdir(parents=True); (root/split/"labels").mkdir(parents=True)

rng = np.random.default_rng(0)
def make_image(path_img, path_lbl, n_fruit):
    W = H = 256
    img = Image.new("RGB", (W, H), (60, 110, 60))          # leafy background
    d = ImageDraw.Draw(img)
    for _ in range(40):                                     # leaf texture
        x, y = rng.integers(0, W, 2); r = rng.integers(6, 20)
        d.ellipse([x-r, y-r, x+r, y+r], fill=(45+int(rng.integers(0,40)), 95, 50))
    lines = []
    for _ in range(n_fruit):
        r = int(rng.integers(12, 20))
        cx, cy = rng.integers(r+2, W-r-2), rng.integers(r+2, H-r-2)
        d.ellipse([cx-r, cy-r, cx+r, cy+r], fill=(200, 40, 35))   # red pepper
        # YOLO label: class cx cy w h, all normalised 0-1
        lines.append(f"0 {cx/W:.6f} {cy/H:.6f} {2*r/W:.6f} {2*r/H:.6f}")
    img.save(path_img); path_lbl.write_text("\n".join(lines))
    return n_fruit

truth = {}
for split, k in [("train", 40), ("val", 12)]:
    for i in range(k):
        n = int(rng.integers(1, 7))
        truth[f"{split}/{i}"] = make_image(root/split/"images"/f"{i}.jpg",
                                           root/split/"labels"/f"{i}.txt", n)
DATA_YAML = root/"data.yaml"
DATA_YAML.write_text(yaml.safe_dump({
    "path": str(root.resolve()), "train": "train/images", "val": "val/images",
    "names": {0: "pepper"}}))
print(DATA_YAML.read_text())
print("train images:", len(glob.glob(str(root/'train/images/*.jpg'))),
      "| val images:", len(glob.glob(str(root/'val/images/*.jpg'))))
Image.open(root/"train/images/0.jpg")

## 4. Starter code from the course

Reference only — the shortest form of today's idea. It assumes variables that the
next section defines properly, so read it, don't run it. Section 5 is the runnable
version.

```python
from ultralytics import YOLO
m=YOLO("yolov8n.pt")
m.train(data="data.yaml",epochs=20,imgsz=640)
print("fruit count:",len(m.predict("test.jpg")[0].boxes))
```

## 5. Worked solution

In [ ]:
# ---- Fine-tune from pre-trained weights (not from scratch) ------------------
model = YOLO("yolov8n.pt")
results = model.train(data=str(DATA_YAML), epochs=20, imgsz=256, batch=8,
                      project="runs", name="pepper", exist_ok=True, verbose=False)
print("\nbest weights:", results.save_dir, "/weights/best.pt")

# ---- Validate ----------------------------------------------------------------
best = YOLO(str(Path(results.save_dir)/"weights"/"best.pt"))
metrics = best.val(data=str(DATA_YAML), verbose=False)
print(f"\nmAP50 {metrics.box.map50:.3f} | mAP50-95 {metrics.box.map:.3f} "
      f"| precision {metrics.box.mp:.3f} | recall {metrics.box.mr:.3f}")

# ---- Counting as a yield proxy ----------------------------------------------
rows = []
for p in sorted(glob.glob(str(root/"val/images/*.jpg"))):
    i = Path(p).stem
    pred = len(best(p, conf=0.35, verbose=False)[0].boxes)
    rows.append((i, truth[f"val/{i}"], pred))
print("\nimage  hand-counted  detected")
for i, t, p in rows:
    flag = "" if t == p else "   <- off by %+d" % (p - t)
    print(f"{i:>5} {t:>13} {p:>9}{flag}")
t_arr = np.array([t for _, t, _ in rows]); p_arr = np.array([p for _, _, p in rows])
print(f"\ntotal hand-counted {t_arr.sum()} | total detected {p_arr.sum()}")
print(f"mean absolute error {np.abs(t_arr-p_arr).mean():.2f} fruit/image")
if t_arr.std() > 0 and p_arr.std() > 0:
    print(f"correlation with hand counts {np.corrcoef(t_arr, p_arr)[0,1]:.3f}")
print("\nCounts are an ESTIMATE: hidden fruit behind leaves is never detected.")
print("Calibrate against hand-counted plots before quoting a yield figure.")

# ---- Look at the predictions -------------------------------------------------
fig, ax = plt.subplots(1, 4, figsize=(13, 3.4))
for a, p in zip(ax, sorted(glob.glob(str(root/"val/images/*.jpg")))[:4]):
    r = best(p, conf=0.35, verbose=False)[0]
    a.imshow(r.plot()[:, :, ::-1]); a.axis("off")
    a.set_title(f"{len(r.boxes)} detected / {truth['val/'+Path(p).stem]} real", fontsize=9)
plt.tight_layout(); plt.show()

## 6. Your turn

1. Train for 5 epochs instead of 20 and compare mAP50 — how much does fine-tuning need?
2. Add a second class (unripe pepper, green) to the generator, data.yaml and training.
3. Export the best weights with `best.export(format='onnx')` for deployment off-GPU.

In [ ]:
# TODO 1: fewer epochs
# YOLO("yolov8n.pt").train(data=str(DATA_YAML), epochs=5, imgsz=256, name="pepper5")

# TODO 2: two classes -- remember names: {0: "ripe", 1: "unripe"} in data.yaml

# TODO 3: export
# print(best.export(format="onnx"))

## 7. Dataset and references

**Dataset:** [Roboflow Universe: search fruit / pepper / weed detection](https://universe.roboflow.com)

**References**

- [Roboflow Universe](https://universe.roboflow.com)
- [Ultralytics training guide](https://docs.ultralytics.com/modes/train/)

**Course material:** Roboflow: train YOLOv8 on a custom dataset

---
**Next:** Day 17 — Attention Mechanics & Vision Transformers